In [2]:
!pip install -q ultralytics huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 8.1 MB/s eta 0:00:00


In [3]:
import os

if not os.path.exists('CUB_200_2011'):
    !wget -q https://data.caltech.edu/records/65de6-vp158/files/CUB_200_2011.tgz
    !tar -xzf CUB_200_2011.tgz
    print("Downloaded and extracted.")
else:
    print("Already present.")

DATA_ROOT = 'CUB_200_2011'

# Verify
!file CUB_200_2011.tgz

Downloaded and extracted.
CUB_200_2011.tgz: gzip compressed data, last modified: Thu Nov  3 19:00:23 2011, from Unix, original size modulo 2^32 1259479040 gzip compressed data, unknown method, has CRC, was "", encrypted, from FAT filesystem (MS-DOS, OS/2, NT), original size modulo 2^32 1259479040


In [4]:
import pandas as pd
from PIL import Image

images = pd.read_csv(f'{DATA_ROOT}/images.txt', sep=' ', names=['img_id', 'filepath'])
split = pd.read_csv(f'{DATA_ROOT}/train_test_split.txt', sep=' ', names=['img_id', 'is_train'])
bboxes = pd.read_csv(f'{DATA_ROOT}/bounding_boxes.txt', sep=' ',
                      names=['img_id', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h'])

df = images.merge(split, on='img_id').merge(bboxes, on='img_id')
df['filepath'] = df['filepath'].apply(lambda x: f'{DATA_ROOT}/images/{x}')

train_df = df[df['is_train'] == 1].reset_index(drop=True)
test_df = df[df['is_train'] == 0].reset_index(drop=True)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")
print(train_df[['img_id', 'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h']].head())

Train: 5994, Test: 5794
   img_id  bbox_x  bbox_y  bbox_w  bbox_h
0       2   139.0    30.0   153.0   264.0
1       4   112.0    90.0   255.0   242.0
2       5    70.0    50.0   134.0   303.0
3       7     7.0    75.0   420.0   262.0
4       8    78.0    86.0   333.0   158.0


In [5]:
import os, shutil
from PIL import Image
from tqdm import tqdm

BASE = '/content/yolo_data'

def build_yolo_split(df, split_name, class_id=0):
    img_dir = f'{BASE}/images/{split_name}'
    lbl_dir = f'{BASE}/labels/{split_name}'
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Building {split_name}"):
        img_id = row['img_id']
        src = row['filepath']
        dst_img = f'{img_dir}/{img_id}.jpg'

        if not os.path.exists(dst_img):
            shutil.copy(src, dst_img)

        with Image.open(src) as im:
            W, H = im.size

        x, y, w, h = row['bbox_x'], row['bbox_y'], row['bbox_w'], row['bbox_h']
        xc = min(max((x + w / 2) / W, 0), 1)
        yc = min(max((y + h / 2) / H, 0), 1)
        wn = min(max(w / W, 0), 1)
        hn = min(max(h / H, 0), 1)

        with open(f'{lbl_dir}/{img_id}.txt', 'w') as f:
            f.write(f"{class_id} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

build_yolo_split(train_df, 'train')
build_yolo_split(test_df, 'val_clean')

print("YOLO dataset built.")

Building val_clean: 100%|██████████| 5794/5794 [00:09<00:00, 600.14it/s]

YOLO dataset built.


In [6]:
clean_yaml = f"""
path: {BASE}
train: images/train
val: images/val_clean
names:
  0: bird
"""
with open(f'{BASE}/clean.yaml', 'w') as f:
    f.write(clean_yaml)

print(open(f'{BASE}/clean.yaml').read())


path: /content/yolo_data
train: images/train
val: images/val_clean
names:
  0: bird



In [7]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=f'{BASE}/clean.yaml',
    epochs=15,
    imgsz=640,
    batch=16,
    project='/content/runs',
    name='bird_detector',
    patience=5
)

print("Training complete.")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.154 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_data/clean.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, 

In [8]:
clean_metrics = model.val(data=f'{BASE}/clean.yaml', split='val')

clean_results = {
    'condition': 'clean',
    'mAP50': float(clean_metrics.box.map50),
    'mAP50-95': float(clean_metrics.box.map),
    'precision': float(clean_metrics.box.mp),
    'recall': float(clean_metrics.box.mr),
}
print(clean_results)

Ultralytics 8.4.154 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1111.1±448.1 MB/s, size: 97.3 KB)
val: Scanning /content/yolo_data/labels/val_clean.cache... 5794 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 5794/5794 1.1Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 363/363 7.2it/s 50.1s
                   all       5794       5794      0.986      0.989      0.994       0.86
Speed: 0.9ms preprocess, 2.9ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to /content/runs/detect/val
{'condition': 'clean', 'mAP50': 0.994091573878786, 'mAP50-95': 0.8600920428796496, 'precision': 0.9858920388343634, 'recall': 0.9890095118941845}


In [9]:
import random
from PIL import Image

random.seed(42)
CANVAS = 640
N_SCENES = 300

def crop_bird(row):
    im = Image.open(row['filepath']).convert('RGB')
    x, y, w, h = row['bbox_x'], row['bbox_y'], row['bbox_w'], row['bbox_h']
    return im.crop((x, y, x + w, y + h))

def make_cluttered_scene(base_row, extra_rows):
    base_im = Image.open(base_row['filepath']).convert('RGB')
    W0, H0 = base_im.size
    canvas = base_im.resize((CANVAS, CANVAS))
    sx, sy = CANVAS / W0, CANVAS / H0

    boxes = []
    bx = base_row['bbox_x'] * sx
    by = base_row['bbox_y'] * sy
    bw = base_row['bbox_w'] * sx
    bh = base_row['bbox_h'] * sy
    boxes.append([bx, by, bx + bw, by + bh])

    for erow in extra_rows:
        crop = crop_bird(erow)
        cw, ch = crop.size
        target_max = random.uniform(0.25, 0.55) * CANVAS
        scale = target_max / max(cw, ch)
        new_w, new_h = max(15, int(cw * scale)), max(15, int(ch * scale))
        crop_resized = crop.resize((new_w, new_h))

        px = random.randint(0, max(0, CANVAS - new_w))
        py = random.randint(0, max(0, CANVAS - new_h))
        canvas.paste(crop_resized, (px, py))
        boxes.append([px, py, px + new_w, py + new_h])

    return canvas, boxes

img_dir = f'{BASE}/images/val_cluttered'
lbl_dir = f'{BASE}/labels/val_cluttered'
os.makedirs(img_dir, exist_ok=True)
os.makedirs(lbl_dir, exist_ok=True)

test_records = test_df.to_dict('records')

for i in tqdm(range(N_SCENES), desc="Building cluttered scenes"):
    base_row = random.choice(test_records)
    n_extra = random.choice([1, 1, 2])  # mostly 1 extra bird, sometimes 2
    extra_rows = random.sample(test_records, n_extra)

    canvas, boxes = make_cluttered_scene(base_row, extra_rows)
    canvas.save(f'{img_dir}/scene_{i:04d}.jpg')

    with open(f'{lbl_dir}/scene_{i:04d}.txt', 'w') as f:
        for (x1, y1, x2, y2) in boxes:
            xc = ((x1 + x2) / 2) / CANVAS
            yc = ((y1 + y2) / 2) / CANVAS
            wn = (x2 - x1) / CANVAS
            hn = (y2 - y1) / CANVAS
            xc, yc, wn, hn = [min(max(v, 0), 1) for v in (xc, yc, wn, hn)]
            f.write(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}\n")

print(f"Built {N_SCENES} synthetic cluttered scenes.")

Building cluttered scenes: 100%|██████████| 300/300 [00:08<00:00, 36.71it/s]

Built 300 synthetic cluttered scenes.


In [10]:
cluttered_yaml = f"""
path: {BASE}
train: images/train
val: images/val_cluttered
names:
  0: bird
"""
with open(f'{BASE}/cluttered.yaml', 'w') as f:
    f.write(cluttered_yaml)

cluttered_metrics = model.val(data=f'{BASE}/cluttered.yaml', split='val')

cluttered_results = {
    'condition': 'cluttered_multi_bird',
    'mAP50': float(cluttered_metrics.box.map50),
    'mAP50-95': float(cluttered_metrics.box.map),
    'precision': float(cluttered_metrics.box.mp),
    'recall': float(cluttered_metrics.box.mr),
}
print(cluttered_results)

Ultralytics 8.4.154 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 809.8±508.8 MB/s, size: 46.0 KB)
val: Scanning /content/yolo_data/labels/val_cluttered... 300 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 300/300 1.4Kit/s 0.2s
val: New cache created: /content/yolo_data/labels/val_cluttered.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 4.5it/s 4.3s
                   all        300        705       0.86      0.689      0.775      0.509
Speed: 1.9ms preprocess, 4.4ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to /content/runs/detect/val-2
{'condition': 'cluttered_multi_bird', 'mAP50': 0.7752471524598186, 'mAP50-95': 0.5092844427903314, 'precision': 0.8597721573175915, 'recall': 0.6893617021276596}


In [11]:
import json

all_results = [clean_results, cluttered_results]
results_df = pd.DataFrame(all_results)
print(results_df)

results_df.to_csv('/content/detection_robustness_results.csv', index=False)
with open('/content/detection_robustness_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

# Save the trained model weights to a known path
best_weights = '/content/runs/bird_detector/weights/best.pt'
print("Best weights at:", best_weights)

              condition     mAP50  mAP50-95  precision    recall
0                 clean  0.994092  0.860092   0.985892  0.989010
1  cluttered_multi_bird  0.775247  0.509284   0.859772  0.689362
Best weights at: /content/runs/bird_detector/weights/best.pt


In [12]:
from huggingface_hub import notebook_login, HfApi, create_repo

notebook_login()  # paste your free HF write token when prompted

In [13]:
HF_USERNAME = "dhayal1"
HF_REPO = f"{HF_USERNAME}/bird-detection-cluttered-scenes"

create_repo(HF_REPO, exist_ok=True)

api = HfApi()
api.upload_file(
    path_or_fileobj='/content/runs/bird_detector/weights/best.pt',
    path_in_repo="bird_detector_best.pt",
    repo_id=HF_REPO
)
api.upload_file(
    path_or_fileobj='/content/detection_robustness_results.json',
    path_in_repo="detection_robustness_results.json",
    repo_id=HF_REPO
)

print(f"Uploaded to https://huggingface.co/{HF_REPO}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._detector/weights/best.pt:   9%|8         |  532kB / 6.22MB            

Uploaded to https://huggingface.co/dhayal1/bird-detection-cluttered-scenes
